# Notebook to vizualize retrieval metrics with different parameters

Firstly compare embeddings models and then focus on all_mini with hyperparameters influence

In [ ]:
import json
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from tqdm import tqdm

from CIT.RAGs.utils import VectorBase

from ..utils import compute_precision_recall_1case, compute_retrieval_stats, load_jsonl

# test different embeddings models (not updated, should use code from the next part to speed things up)

In [ ]:
existing_recalls = pd.read_json("existing_recalls.json")
recalls_df = existing_recalls.copy()

In [ ]:
all_questions_folder = "./src/CIT/evaluation/QA_generation/data_by_hand"
questions_with_urls = []
for file in os.listdir(all_questions_folder):
    questions_with_urls += load_jsonl(os.path.join(all_questions_folder, file))

In [ ]:
len(questions_with_urls)

In [ ]:
DIRECTORY = "./src/CIT/documents/confluence_json"
DIR = "./src/CIT/documents/confluence_json_without_root_with_titles"
CHUNK_SIZE = [1500]
CHUNK_OVERLAP = 300
TOP_K = 6
embedding1 = "all-MiniLM-L6-v2"
embedding2 = "Alibaba-NLP/gte-Qwen2-7B-instruct"
embedding3 = "infly/inf-retriever-v1-1.5b"
embedding4 = "infly/inf-retriever-v1"

In [ ]:
if embedding1 not in existing_recalls.model.unique():
    print("Loading all-MiniLM-L6-v2")
    vector_base_all_mini = VectorBase(
        DIR,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embedding_model=embedding1,
    )
    retriever_all_mini = vector_base_all_mini.vectorstore.as_retriever(
        search_kwargs={"k": 15}
    )

if embedding2 not in existing_recalls.model.unique():
    print("Loading Alibaba-NLP/gte-Qwen2-7B-instruct")
    vector_base_qwen7b = VectorBase(
        DIR,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embedding_model=embedding2,
    )
    retriever_qwen7b = vector_base_qwen7b.vectorstore.as_retriever(
        search_kwargs={"k": 15}
    )
if embedding3 not in existing_recalls.model.unique():
    print("Loading infly/inf-retriever-v1-1.5b")
    vector_base_inf1_5b = VectorBase(
        DIR,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embedding_model=embedding3,
    )
    retriever_inf1_5b = vector_base_inf1_5b.vectorstore.as_retriever(
        search_kwargs={"k": 15}
    )
if embedding4 not in existing_recalls.model.unique():
    print("Loading infly/inf-retriever-v1")
    vector_base_inf7b = VectorBase(
        DIR,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        embedding_model=embedding4,
    )
    retriever_inf7b = vector_base_inf7b.vectorstore.as_retriever(
        search_kwargs={"k": 15}
    )

In [ ]:
def compute_topks_recall(retriever, questions, max_top_k=10):
    recalls = []
    for top_k in range(1, max_top_k + 1):
        print(f"top k: {top_k}")
        compressor = FlashrankRerank(top_n=top_k, model="ms-marco-MiniLM-L-12-v2")
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor, base_retriever=retriever
        )
        mean_precision, mean_recall, average_precision, average_recall = (
            compute_retrieval_stats(questions, compression_retriever)
        )
        recalls.append(
            {"k": top_k, "recall": mean_recall, "avg_recall": average_recall}
        )
    return recalls

In [ ]:
recalls_all_mini = compute_topks_recall(retriever_all_mini, max_top_k=10)
recalls_all_mini_df = pd.DataFrame(recalls_all_mini)
recalls_all_mini_df["model"] = embedding1

In [ ]:
recalls_qwen7b = compute_topks_recall(retriever_qwen7b, max_top_k=10)
recalls_qwen7b_df = pd.DataFrame(recalls_qwen7b)
recalls_qwen7b_df["model"] = embedding2

In [ ]:
recalls_inf1_5b = compute_topks_recall(retriever_inf1_5b, max_top_k=10)
recalls_inf1_5b_df = pd.DataFrame(recalls_inf1_5b)
recalls_inf1_5b_df["model"] = embedding3

In [ ]:
recalls_inf1_7b = compute_topks_recall(retriever_inf7b, max_top_k=10)
recalls_inf1_7b_df = pd.DataFrame(recalls_inf1_7b)
recalls_inf1_7b_df["model"] = embedding4

In [ ]:
to_be_concat = []
if embedding1 not in existing_recalls.model.unique():
    to_be_concat.append(recalls_all_mini_df)
if embedding2 not in existing_recalls.model.unique():
    to_be_concat.append(recalls_qwen7b_df)
if embedding3 not in existing_recalls.model.unique():
    to_be_concat.append(recalls_inf1_5b_df)
if embedding4 not in existing_recalls.model.unique():
    to_be_concat.append(recalls_inf1_7b_df)
if len(to_be_concat) > 0:
    recalls_df = pd.concat([existing_recalls] + to_be_concat)
else:
    recalls_df = existing_recalls

fig, ax = plt.subplots(figsize=(10, 5))
ax2 = ax.twinx()
sns.lineplot(data=recalls_df, x="k", y="recall", hue="model", ax=ax)
ax2 = sns.lineplot(
    data=recalls_df, x="k", y="avg_recall", hue="model", ax=ax2, linestyle="--"
)
ax.set_ylabel("Recall")
ax2.set_ylabel("Avg Recall")
ax.set_xlabel("Top K")
ax.set_title("Recall and Avg Recall for different Top K values")
ax.legend(title="Recall", loc="upper left")
ax2.legend(title="AVg recall", loc="lower right")
plt.savefig("images/recalls.png")
plt.show()

In [ ]:
recalls_df.to_json("existing_recalls.json", orient="records")

# All mini v2 specific

In [ ]:
vector_base_all_mini = VectorBase(
    DIR, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, embedding_model=embedding1
)
retriever = vector_base_all_mini.vectorstore.as_retriever(search_kwargs={"k": 15})

In [ ]:
questions_with_urls = load_jsonl(
    "./src/CIT/training/data/all_data_copy.jsonl"
)

In [ ]:
unique_ids = set()
unique_questions = []
for i in range(len(questions_with_urls)):
    if questions_with_urls[i]["question_rephrased_id"] not in unique_ids:
        unique_ids.add(questions_with_urls[i]["question_rephrased_id"])
        unique_questions.append(questions_with_urls[i])

In [ ]:
def retrieve_with_scores(query: str, compression_retriever, mapping_id_urls):
    """Retrieve documents related to a user query to help answer a question.
    Args:
        query (str): user question or query.
    Returns:
        Tuple[str, List[Document]]: retrieved documents."""
    retrieved_docs = compression_retriever.invoke(query)
    retrieved_docs_urls_with_scores = [
        (doc.metadata["url"], doc.metadata["relevance_score"]) for doc in retrieved_docs
    ]
    # add url of best parent if more than 2 times
    parents_id = Counter([doc.metadata["parent"] for doc in retrieved_docs])
    if parents_id.most_common(1)[0][1] > 2:
        primary_parent_id = parents_id.most_common(1)[0][0]
        sources_id = Counter([doc.metadata["id"] for doc in retrieved_docs])
        if (primary_parent_id in mapping_id_urls) and (
            primary_parent_id not in sources_id
        ):
            retrieved_docs_urls_with_scores.append(
                (mapping_id_urls[primary_parent_id], 1)
            )
    scale = np.max([doc.metadata["relevance_score"] for doc in retrieved_docs])

    return retrieved_docs_urls_with_scores, scale

In [ ]:
with open("./src/CIT/documents/mappings/mapping_id_urls.json", "r") as f:
    mapping_id_urls = json.load(f)

In [ ]:
def compute_topks_recall(
    retriever, questions, max_top_k=10, max_threshold_score=0.5, step_threshold=0.05
):
    thresholds_list = list(
        np.arange(0, max_threshold_score + step_threshold, step_threshold)
    )
    recalls = {
        k: {threshold: [] for threshold in thresholds_list}
        for k in range(1, max_top_k + 1)
    }
    # recalls={k:[] for k in range(1,max_top_k+1)}
    TPs = {
        k: {threshold: 0 for threshold in thresholds_list}
        for k in range(1, max_top_k + 1)
    }
    # TPs={k:0 for k in range(1,max_top_k+1)}
    FNs = {
        k: {threshold: 0 for threshold in thresholds_list}
        for k in range(1, max_top_k + 1)
    }
    # FNs={k:0 for k in range(1,max_top_k+1)}
    compressor = FlashrankRerank(top_n=max_top_k, model="ms-marco-MiniLM-L-12-v2")
    base_retriever = vector_base_all_mini.vectorstore.as_retriever(
        search_kwargs={"k": 50}
    )
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=base_retriever
    )
    for question in tqdm(questions, total=len(questions)):
        query = question["question"]
        expected_docs_urls = question["urls"]
        retrieved_docs_urls_with_scores, scale_score = retrieve_with_scores(
            query, compression_retriever, mapping_id_urls
        )
        for k in range(1, max_top_k + 1):
            for threshold in thresholds_list:
                real_threshold = threshold * scale_score

                for doc in retrieved_docs_urls_with_scores[:k]:
                    try:
                        if doc[1] >= real_threshold:
                            pass
                    except:
                        print(f"Error in doc: {doc}")
                        print(f"Question: {question}")
                        print(f"Threshold: {threshold}")
                        print(f"Scale score: {scale_score}")
                        print(f"Real threshold: {real_threshold}")
                        continue

                retrieved_docs_urls = [
                    doc[0]
                    for doc in retrieved_docs_urls_with_scores[:k]
                    if doc[1] >= real_threshold
                ]

                results = {
                    "urls": expected_docs_urls,
                    "RAG_confluence_urls": retrieved_docs_urls,
                }
                if "facultative_urls" in question:
                    results["facultative_urls"] = question["facultative_urls"]

                precision, recall, TP, FP, FN = compute_precision_recall_1case(results)
                recalls[k][threshold].append(recall)
                TPs[k][threshold] += TP
                FNs[k][threshold] += FN

    avg_recalls = {}
    for k in range(1, max_top_k + 1):
        avg_recalls[k] = {}
        for threshold in thresholds_list:
            recalls[k][threshold] = np.mean(recalls[k][threshold])
            avg_recall = TPs[k][threshold] / (TPs[k][threshold] + FNs[k][threshold])
            avg_recalls[k][threshold] = avg_recall
    return recalls, avg_recalls

In [ ]:
max_top_k = 50
max_threshold_score = 0.5
step_threshold = 0.05
recalls, avg_recalls = compute_topks_recall(
    retriever,
    unique_questions,
    max_top_k=max_top_k,
    max_threshold_score=max_threshold_score,
    step_threshold=step_threshold,
)

In [ ]:
rows = []
for k, thresholds in recalls.items():
    for threshold, recall in thresholds.items():
        avg_recall = avg_recalls[k][threshold]
        rows.append(
            {"k": k, "threshold": threshold, "recall": recall, "avg_recall": avg_recall}
        )
recalls_df = pd.DataFrame(rows)

Threshold 0:

In [ ]:
# plot the results with 2 y axes for recall and average recall
# recalls_df=pd.DataFrame(recalls)
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
data = recalls_df[recalls_df["threshold"] == 0]
sns.lineplot(data=data, x="k", y="recall", ax=ax1, label="Recall", color="blue")
sns.lineplot(
    data=data, x="k", y="avg_recall", ax=ax2, label="Average Recall", color="orange"
)
plt.vlines(
    x=6, ymin=0, ymax=1, color="gray", alpha=0.5, linestyle="--", label="Top K=6"
)
ax1.set_ylabel("Mean Recall")
ax2.set_ylabel("Average Recall")
ax1.set_xlabel("Top K")
ax1.set_title("Recall vs Top K")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
ax1.set_ylim(0.6, 1)
ax2.set_ylim(0.6, 1)
ax1.set_yticks(np.arange(0.6, 1, 0.05))
ax2.set_yticks(np.arange(0.6, 1, 0.05))


plt.show()

compare thresholds:

In [ ]:
thresholds_list = list(
    np.arange(0, max_threshold_score + step_threshold, step_threshold)
)
fig, ax1 = plt.subplots()
for threshold in thresholds_list:
    data = recalls_df[recalls_df["threshold"] == threshold]
    sns.lineplot(
        data=data, x="k", y="recall", ax=ax1, label=f"Threshold {threshold:.2f}"
    )
ax1.set_ylabel("Mean Recall")
ax1.set_xlabel("Top K")
ax1.set_title("Recall vs Top K")
plt.vlines(
    x=6, ymin=0, ymax=1, color="gray", alpha=0.5, linestyle="--", label="Top K=6"
)
ax1.legend(loc="best")
ax1.set_ylim(0.6, 1)
ax1.set_yticks(np.arange(0.6, 1, 0.05))
plt.savefig("./src/CIT/evaluation/viz/images/recall_topk_thresholds.png")
plt.show()

In [ ]:
thresholds_list = list(
    np.arange(0, max_threshold_score + step_threshold, step_threshold)
)
fig, ax1 = plt.subplots()
for threshold in thresholds_list:
    data = recalls_df[recalls_df["threshold"] == threshold]
    sns.lineplot(
        data=data, x="k", y="recall", ax=ax1, label=f"Threshold {threshold:.2f}"
    )
ax1.set_ylabel("Mean Recall")
ax1.set_xlabel("Top K")
ax1.set_title("Recall vs Top K")
ax1.legend(loc="best")
ax1.set_ylim(0.6, 1)
ax1.set_yticks(np.arange(0.6, 1, 0.05))
plt.vlines(
    x=6, ymin=0, ymax=1, color="gray", alpha=0.5, linestyle="--", label="Top K=6"
)
plt.show()

# Appendix, manual inspection

### Look recall on basic questions

In [ ]:
basic_questions = [qu for qu in questions_with_urls if len(qu["urls"]) < 2]

In [ ]:
recalls_basic = compute_topks_recall(retriever, basic_questions, max_top_k=10)

In [ ]:
# plot the results with 2 y axes for recall and average recall
recalls_basic_df = pd.DataFrame(recalls_basic)
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
sns.lineplot(
    data=recalls_basic_df, x="k", y="recall", ax=ax1, label="Recall", color="blue"
)
sns.lineplot(
    data=recalls_basic_df,
    x="k",
    y="avg_recall",
    ax=ax2,
    label="Average Recall",
    color="orange",
)
ax1.set_ylabel("Mean Recall")
ax2.set_ylabel("Average Recall")
ax1.set_xlabel("Top K")
ax1.set_title(
    "Recall vs Top K for basic question (<=1 url to be retrieved) \n(overlap of the 2 lines)"
)
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.show()

### For complex questions (>=2 urls to be retrieved)

In [ ]:
complex_questions = [qu for qu in questions_with_urls if len(qu["urls"]) > 1]
recalls_complex = compute_topks_recall(retriever, complex_questions, max_top_k=10)

In [ ]:
# plot the results with 2 y axes for recall and average recall
recalls_complex_df = pd.DataFrame(recalls_complex)
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
sns.lineplot(
    data=recalls_complex_df, x="k", y="recall", ax=ax1, label="Recall", color="blue"
)
sns.lineplot(
    data=recalls_complex_df,
    x="k",
    y="avg_recall",
    ax=ax2,
    label="Average Recall",
    color="orange",
)
ax1.set_ylabel("Recall")
ax2.set_ylabel("Average Recall")
ax1.set_xlabel("Top K")
ax1.set_title("Recall vs Top K for complex question (>=2 urls to be retrieved)")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.show()